# datacleaner Demo Notebook

This notebook demonstrates how to use `analyze` and `clean` on a realistic messy dataset.

In [ ]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
import io
import sys

import pandas as pd

# Make local package importable when notebook is run from examples/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from datacleaner import analyze, clean

## Create a Messy Dataset

The dataset below intentionally includes:
- Missing values
- Duplicate rows
- Incorrect datatypes (numeric-like values stored as text)
- Outliers

In [ ]:
data = {
    "customer_id": [
        "C001", "C002", "C003", "C004", "C005", "C006", "C007", "C008", "C009", "C010",
        "C011", "C012", "C013", "C014", "C015", "C016", "C017", "C018", "C019", "C020",
    ],
    "age": [
        "25", "31", "29", None, "42", "38", "35", "unknown", "41", "27",
        "33", "29", None, "36", "120", "28", "30", "32", "26", "300",
    ],
    "income": [
        "50000", "62000", "58000", "54000", None, "71000", "68000", "65000", "60000", "56000",
        "59000", "61000", "62000", "63000", "1000000", "57000", "58000", "59000", "60000", "-50000",
    ],
    "city": [
        "New York", "Chicago", None, "Houston", "Seattle", "Chicago", "Austin", "Austin", "Miami", None,
        "Denver", "Denver", "Seattle", "Houston", "Austin", "Chicago", "New York", None, "Miami", "Miami",
    ],
    "signup_date": [
        "2024-01-10", "2024/02/11", "2024-03-12", "not_a_date", "2024-05-01", None, "2024-07-15", "2024-08-20", "2024-09-10", "2024-10-05",
        "2024-11-11", "2024-12-12", None, "2025-01-01", "2025-02-02", "2025-03-03", "2025-04-04", "2025-05-05", "2025-06-06", "2025-07-07",
    ],
    "notes": [
        None, None, "late payment", None, None, "vip", None, None, None, "call back",
        None, None, None, None, None, None, "prefers email", None, None, None,
    ],
}

df = pd.DataFrame(data)

# Add exact duplicate rows.
df = pd.concat([df, df.iloc[[1, 4, 7]]], ignore_index=True)

df.shape

## 1. Display Raw Dataset

We inspect head, info, and describe before cleaning.

In [ ]:
print("Shape:", df.shape)
display(df.head())

buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

display(df.describe(include="all"))

## 2. Run `analyze(df)`

This gives a quick pre-cleaning quality summary and warnings.

In [ ]:
analysis_report = analyze(df)
pprint(analysis_report, sort_dicts=False)

## 3. Run `clean(df, verbose=True)`

Verbose mode prints step-by-step actions taken by the cleaning pipeline.

In [ ]:
cleaned_df, cleaning_report = clean(df, verbose=True)

## 4. Display Cleaned Dataset

Check the cleaned dataframe after the pipeline runs.

In [ ]:
print("Shape:", cleaned_df.shape)
display(cleaned_df.head())

buffer = io.StringIO()
cleaned_df.info(buf=buffer)
print(buffer.getvalue())

display(cleaned_df.describe(include="all"))

## 5. Key Differences and Final Report

Summarize what changed from before to after cleaning.

In [ ]:
rows_before, cols_before = df.shape
rows_after, cols_after = cleaned_df.shape

rows_removed = rows_before - rows_after
columns_dropped = cols_before - cols_after

print("Before vs After")
print(f"Rows: {rows_before} -> {rows_after} (removed: {rows_removed})")
print(f"Columns: {cols_before} -> {cols_after} (dropped: {columns_dropped})")

print("\nCleaning report:")
pprint(cleaning_report, sort_dicts=False)